# Sesión 1 — Fundamentos de ML, Regresión y Clasificación

Código de los conceptos de la sesión y el ejercicio para practicarlos.

## 1. Introducción al Machine Learning

### 1.1 IA → ML → Deep Learning

Estos tres términos se usan muchas veces como sinónimos, pero forman
una jerarquía de contención:

- **Inteligencia Artificial (IA)**: el campo más amplio; cualquier
  técnica que permita a una máquina imitar comportamiento inteligente
  (razonar, planear, percibir, decidir).
- **Machine Learning (ML)**: un subconjunto de la IA. En vez de
  programar reglas explícitas, el sistema **aprende patrones a partir
  de datos** para tomar decisiones o hacer predicciones.
- **Deep Learning (DL)**: un subconjunto del ML que usa redes
  neuronales con varias capas (arquitecturas "profundas") para aprender
  representaciones cada vez más abstractas de los datos, especialmente
  útil en dominios como visión por computador, texto y audio.

```
IA  ⊃  Machine Learning  ⊃  Deep Learning
```

### 1.2 Aplicaciones de ML

ML ya está presente en tareas muy cotidianas:

- Recomendación de contenido (Netflix, Spotify, e-commerce).
- Detección de fraude en transacciones financieras.
- Diagnóstico asistido en imágenes médicas.
- Mantenimiento predictivo en plantas industriales.
- Precificación dinámica y estimación de valor de activos (p. ej.
  precios de vivienda, el problema que usaremos hoy).
- Modelos de riesgo crediticio y scoring.

### 1.3 Pipeline de un proyecto de ML

Un proyecto de ML no es solo "entrenar un modelo": es un proceso con
varias etapas, típicamente iterativo (no estrictamente lineal):

1. **Data Collection** — identificar y recolectar las fuentes de datos
   relevantes para el problema de negocio.
2. **Data Cleaning and Preparation** — tratar valores faltantes,
   errores, outliers, formatos inconsistentes; construir variables
   (feature engineering).
3. **Modeling and Optimization** — seleccionar y entrenar modelos,
   ajustar hiperparámetros.
4. **Model Evaluation** — medir el desempeño con métricas apropiadas,
   sobre datos que el modelo no vio en entrenamiento.
5. **Model Deployment and Monitoring** — poner el modelo en producción
   y monitorear su desempeño en el tiempo (los datos del mundo real
   cambian, y el modelo puede degradarse).

Hoy nos enfocamos principalmente en las etapas 2, 3 y 4 para un
problema de regresión.

## 2. Aprendizaje supervisado vs. no supervisado

- **Aprendizaje no supervisado**: solo tenemos variables de entrada
  $X$, **no hay una variable objetivo conocida**. El objetivo es
  descubrir estructura o patrones en los datos (por ejemplo, agrupar
  clientes similares con *clustering*, o reducir dimensionalidad con
  PCA). No hay una "respuesta correcta" contra la cual comparar.

- **Aprendizaje supervisado**: tenemos variables de entrada $X$ **y**
  una variable objetivo conocida $Y$ para cada observación de
  entrenamiento. El modelo aprende la relación entre $X$ y $Y$ para
  luego predecir $Y$ en observaciones nuevas donde no la conocemos.

  - Si $Y$ es **numérica y continua** → problema de **regresión**
    (el tema de hoy: precio de una vivienda, temperatura, demanda,
    etc.).
  - Si $Y$ es **categórica** → problema de **clasificación** (lo
    veremos en una sesión futura): ¿el cliente se va a fugar sí/no?,
    ¿la transacción es fraude sí/no?

En el dataset de Ames Housing, `price` es una variable numérica y
continua conocida para cada casa del histórico, así que este es un
problema **supervisado de regresión**.

## 3. Metodología del aprendizaje supervisado

Un proyecto de aprendizaje supervisado sigue, en términos generales,
esta secuencia:

1. **Análisis exploratorio (EDA)** — entender la forma de los datos:
   distribuciones, relaciones entre variables, valores atípicos.
2. **Calidad de datos** — identificar y tratar valores faltantes,
   inconsistencias, duplicados, errores de captura.
3. **Definición de métricas** — decidir *antes* de modelar cómo se va
   a medir el éxito (por ejemplo, RMSE para regresión). Esto evita
   elegir la métrica que más le convenga al modelo *después* de verlo.
4. **Partición del conjunto de datos** — separar en train / validation
   / test (lo vemos en detalle en la siguiente sección).
5. **Selección de variables** — decidir qué variables de entrada usar,
   descartando las irrelevantes, redundantes o que generan fuga de
   información (*data leakage*).
6. **Ajuste y selección de modelos** — entrenar distintos algoritmos
   y/o configuraciones, comparándolos de forma justa.
7. **Estandarización** — muchos algoritmos (regresión regularizada,
   KNN, redes neuronales, SVM) son sensibles a la escala de las
   variables; conviene estandarizar o normalizar.
8. **Evaluación de modelos** — medir el desempeño final sobre datos que
   el modelo nunca vio, usando las métricas definidas en el paso 3.

Vamos a recorrer esta metodología aplicada al dataset de Ames Housing.

## 4. Cargando y explorando el dataset Ames Housing

### El problema

Una inmobiliaria quiere una herramienta que estime el **precio de venta
de una casa** a partir de sus características (área, calidad de
construcción, año, número de baños, etc.), para dar una valoración de
referencia rápida a compradores y vendedores, sin depender de un avalúo
manual para cada propiedad.

### El dataset

**Ames Housing**: cerca de 2.900 ventas de casas residenciales en Ames,
Iowa (EE. UU.), con más de 80 variables por vivienda (área, calidad de
materiales, año de construcción, garaje, sótano, etc.). Es uno de los
datasets de referencia más usados para enseñar regresión, con variables
reales: algunas categóricas, otras con valores faltantes.

Cargamos el dataset directamente desde una URL pública (sin descargar
archivos a mano).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

url = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/openintro/ames.csv"
df = pd.read_csv(url)
print(df.shape)
df.head()

Confirmamos el nombre real de la variable objetivo (precio de venta)
inspeccionando las columnas, en vez de asumirlo:

In [ ]:
precio_candidatas = [c for c in df.columns if "price" in c.lower()]
print("Columnas candidatas a variable objetivo:", precio_candidatas)

target_col = "price"
df[target_col].describe()

In [ ]:
df.info()

### Calidad de datos: valores nulos

El dataset original tiene 83 columnas, muchas categóricas con bastantes
nulos (por ejemplo `Pool.QC` o `Alley`, donde el nulo en realidad
significa "no tiene piscina" / "no tiene callejón"). Para esta primera
sesión trabajaremos con un subconjunto de variables **numéricas** con
pocos o ningún valor faltante, y dejamos el tratamiento más cuidadoso
de variables categóricas y nulos para sesiones futuras.

In [ ]:
nulos = df.isna().sum().sort_values(ascending=False)
nulos[nulos > 0].head(15)

In [ ]:
features = [
    "area",            # área habitable
    "Overall.Qual",    # calidad general de materiales y acabados (1-10)
    "Year.Built",       # año de construcción
    "Total.Bsmt.SF",    # área total de sótano
    "Garage.Cars",      # capacidad del garaje en autos
    "Full.Bath",        # baños completos
    "Bedroom.AbvGr",    # habitaciones sobre el nivel del suelo
    "Lot.Area",          # área del lote
    "TotRms.AbvGrd",     # total de habitaciones sobre el nivel del suelo
]

datos = df[features + [target_col]].dropna().reset_index(drop=True)
print("Filas antes de eliminar nulos en las variables elegidas:", len(df))
print("Filas después:", len(datos))
datos.describe()

### Distribución de la variable objetivo

Antes de modelar conviene mirar cómo se distribuye `price`: si está muy
sesgada, algunos modelos (y algunas métricas) se ven afectados por los
valores extremos.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(datos[target_col], bins=40, color="steelblue", edgecolor="white")
axes[0].set_title("Distribución de price")
axes[0].set_xlabel("price (USD)")
axes[0].set_ylabel("frecuencia")

axes[1].scatter(datos["area"], datos[target_col], alpha=0.4, s=12, color="steelblue")
axes[1].set_title("price vs. area")
axes[1].set_xlabel("area (pies cuadrados)")
axes[1].set_ylabel("price (USD)")

plt.tight_layout()
plt.show()

La distribución de `price` tiene cola derecha (algunas casas muy
costosas): esto es típico en variables de precio, y es relevante porque
las métricas basadas en errores al cuadrado (MSE/RMSE) van a ser más
sensibles a esos valores extremos que MAE, como veremos más adelante.

## 5. Partición de datos: train / validation / test

Para evaluar un modelo de forma **justa** necesitamos medir su
desempeño sobre datos que **nunca vio durante el entrenamiento**. Si
evaluamos con los mismos datos que usamos para ajustar el modelo,
corremos el riesgo de una **fuga de información** (*data leakage*): el
modelo "memoriza" en vez de "generalizar", y la métrica que reportamos
es demasiado optimista frente a lo que pasará con datos nuevos en
producción.

Por eso partimos los datos en (al menos) tres conjuntos:

- **Train**: se usa para *ajustar* los parámetros del modelo.
- **Validation**: se usa para *comparar modelos o configuraciones*
  (por ejemplo, elegir hiperparámetros) sin tocar el test.
- **Test**: se usa **una sola vez**, al final, para estimar cómo se
  comportará el modelo con datos nuevos del mundo real.

Una partición típica es 60/20/20 o 70/15/15, dependiendo del tamaño del
dataset. Hoy usaremos 60% train, 20% validation, 20% test.

In [ ]:
X = datos[features]
y = datos[target_col]

# Primero separamos test (20%) del resto (80%)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

# Del 80% restante, separamos validation (25% de ese 80% = 20% del total)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=RANDOM_STATE
)

print("Train:     ", X_train.shape)
print("Validation:", X_val.shape)
print("Test:      ", X_test.shape)

## 6. El problema de regresión

En un problema de regresión asumimos que existe una relación (no
necesariamente lineal) entre las variables de entrada $X$ y la variable
objetivo $Y$, de la forma:

$$Y = f(X) + \varepsilon$$

donde:

- $f$ es una función **desconocida** que describe la relación
  sistemática entre $X$ y $Y$.
- $\varepsilon$ es un término de error aleatorio, con media cero,
  independiente de $X$, que recoge todo lo que $X$ no puede explicar.

El objetivo del aprendizaje supervisado en regresión es **estimar
$f$** a partir de los datos observados, obteniendo una función
$\hat{f}$ tal que $\hat{Y} = \hat{f}(X)$ sea una buena aproximación de
$Y$.

### ¿Para qué queremos estimar $f$?

1. **Predicción**: dado un nuevo punto $X$ (una casa que no está en el
   histórico), predecir $\hat{Y}$ (su precio esperado).
2. **Entender la importancia de las variables**: ¿qué variables de
   entrada son las que más influyen sobre $Y$? (por ejemplo, ¿el área
   pesa más que el año de construcción?).
3. **Interpretar cómo cada componente de $X$ afecta a $Y$**: por
   ejemplo, ¿cuánto sube el precio esperado por cada punto adicional de
   calidad general (`Overall.Qual`), manteniendo lo demás constante?

### Complejidad del modelo (idea intuitiva)

Al estimar $\hat{f}$ podemos elegir modelos con distinta
**flexibilidad**:

- Un modelo **poco flexible** (por ejemplo, una regresión lineal
  simple) puede no capturar relaciones complejas o no lineales entre
  $X$ y $Y$: se queda "corto".
- Un modelo **muy flexible** (por ejemplo, un árbol muy profundo) puede
  ajustarse demasiado a las particularidades de los datos de
  entrenamiento, incluyendo el ruido $\varepsilon$, y no generalizar
  bien a datos nuevos.
- Existe un punto "intermedio" donde el modelo captura la señal real de
  $f$ sin sobreajustarse al ruido.

Esta tensión se llama el **balance sesgo-varianza**, y la
estudiaremos en profundidad en la Sesión 2. Por ahora, la vamos a
**ilustrar de forma intuitiva** comparando dos modelos: uno poco
flexible (Regresión Lineal) y uno más flexible (Random Forest).

### Modelo 1: Regresión Lineal (poco flexible)

La regresión lineal asume que $f$ tiene la forma:

$$\hat{Y} = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \dots + \beta_p X_p$$

Es un modelo simple, fácil de interpretar (cada $\beta_i$ nos dice
cuánto cambia $\hat{Y}$ por cada unidad de $X_i$, manteniendo lo demás
constante), pero limitado a relaciones lineales.

In [ ]:
modelo_lineal = LinearRegression()
modelo_lineal.fit(X_train, y_train)

coeficientes = pd.Series(modelo_lineal.coef_, index=features).sort_values(key=abs, ascending=False)
print("Intercepto:", modelo_lineal.intercept_)
coeficientes

### Modelo 2: Random Forest (más flexible)

Un Random Forest es un conjunto (*ensemble*) de árboles de decisión que
puede capturar relaciones no lineales e interacciones entre variables
sin que tengamos que especificarlas manualmente. Es un modelo mucho más
flexible que la regresión lineal.

In [ ]:
modelo_rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
modelo_rf.fit(X_train, y_train)

importancias = pd.Series(modelo_rf.feature_importances_, index=features).sort_values(ascending=False)
importancias

## 7. Métricas de desempeño para regresión

Sea $y_i$ el valor real, $\hat{y}_i$ el valor predicho, y $n$ el número
de observaciones.

### MAE — Mean Absolute Error

$$\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$

Promedio del error absoluto, en las mismas unidades que $Y$. Es
**robusto a outliers**: un error muy grande pesa proporcionalmente a su
tamaño, no al cuadrado.

### MSE — Mean Squared Error

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

Promedio del error al cuadrado. Al elevar al cuadrado, **penaliza más
los errores grandes** que los pequeños, pero queda en unidades al
cuadrado (difícil de interpretar directamente).

### RMSE — Root Mean Squared Error

$$\text{RMSE} = \sqrt{\text{MSE}}$$

Es la raíz del MSE, así que vuelve a las unidades originales de $Y$.
Sigue **penalizando más los errores grandes** que el MAE, y es de las
métricas más usadas en la práctica (y la que usaremos para calificar el
assignment de esta sesión).

### $R^2$ — Coeficiente de determinación

$$R^2 = 1 - \frac{\sum_{i=1}^{n} (y_i - \hat{y}_i)^2}{\sum_{i=1}^{n} (y_i - \bar{y})^2}$$

Mide la **proporción de la varianza de $Y$ explicada por el modelo**,
comparado contra un modelo trivial que siempre predice el promedio
$\bar{y}$. $R^2 = 1$ es ajuste perfecto; $R^2 = 0$ equivale a predecir
siempre el promedio; puede ser negativo si el modelo es peor que
predecir el promedio.

### MAPE — Mean Absolute Percentage Error

$$\text{MAPE} = \frac{1}{n} \sum_{i=1}^{n} \left| \frac{y_i - \hat{y}_i}{y_i} \right|$$

Expresa el error como **porcentaje** del valor real, lo cual facilita
la interpretación de negocio ("el modelo se equivoca en promedio un
8%"). Su gran limitación: **falla (se dispara o es indefinido) cuando
$y_i$ está cerca de cero**, porque se divide por $y_i$.

### ¿Cuál usar?

No hay una única respuesta correcta; depende del problema:

- Si hay outliers y no quieres que dominen la métrica → **MAE**.
- Si los errores grandes son especialmente costosos en tu problema de
  negocio → **MSE/RMSE**.
- Si quieres comunicar qué tan bien explicas la varianza total →
  **$R^2$**.
- Si quieres comunicar el error en términos relativos/porcentuales, y
  tu variable objetivo nunca está cerca de cero → **MAPE**.

In [ ]:
def calcular_metricas(y_true, y_pred, nombre_modelo):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return pd.Series(
        {"MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2, "MAPE": mape},
        name=nombre_modelo,
    )

pred_lineal_val = modelo_lineal.predict(X_val)
pred_rf_val = modelo_rf.predict(X_val)

resultados_val = pd.concat(
    [
        calcular_metricas(y_val, pred_lineal_val, "Regresión Lineal"),
        calcular_metricas(y_val, pred_rf_val, "Random Forest"),
    ],
    axis=1,
)
resultados_val.round(2)

**Lectura de resultados (validation):**

- El Random Forest, al ser más flexible, generalmente logra un
  MAE/RMSE menor y un $R^2$ mayor que la Regresión Lineal en este
  conjunto de validación: captura relaciones no lineales que la
  regresión lineal no puede representar.
- Esto **no** significa que "más flexible siempre es mejor": con pocos
  datos o muchas variables, un modelo muy flexible puede sobreajustar
  (memorizar el ruido de train) y desempeñarse peor en datos nuevos.
  Este balance (sesgo-varianza) es el tema central de la Sesión 2.
- Nota que evaluamos ambos modelos sobre **validation**, no sobre
  **test**: el test lo reservamos para la evaluación final, una vez
  hayamos decidido qué modelo usar.

### Evaluación final sobre el conjunto de test

Supongamos que, tras comparar en validation, decidimos quedarnos con el
Random Forest. La evaluación **final**, la que reportaríamos como el
desempeño esperado del modelo en producción, se hace sobre **test**
(datos que el modelo no vio ni en entrenamiento ni en la selección de
modelo).

In [ ]:
pred_rf_test = modelo_rf.predict(X_test)
metricas_test = calcular_metricas(y_test, pred_rf_test, "Random Forest (test)")
metricas_test.round(2)

## Resumen de conceptos clave — Regresión

- **IA ⊃ ML ⊃ Deep Learning**: el ML aprende patrones de datos en vez
  de seguir reglas explícitas; el Deep Learning es ML con redes
  neuronales profundas.
- El pipeline de un proyecto de ML incluye: recolección de datos,
  limpieza y preparación, modelado y optimización, evaluación, y
  despliegue con monitoreo.
- **No supervisado**: solo $X$, se buscan patrones. **Supervisado**: $X$
  y $Y$ conocidos; si $Y$ es numérico y continuo, es un problema de
  **regresión** — si es una etiqueta de clase, es un problema de
  **clasificación** (lo que viene a continuación).
- La metodología de un proyecto supervisado sigue, en general: EDA →
  calidad de datos → definición de métricas → partición de datos →
  selección de variables → ajuste/selección de modelos →
  estandarización → evaluación.
- Partir los datos en **train/validation/test** es indispensable para
  evitar fuga de información y obtener una estimación honesta del
  desempeño del modelo en datos nuevos.
- En regresión, buscamos estimar una función $f$ desconocida tal que
  $Y \approx f(X)$; la complejidad del modelo (flexibilidad) es una
  decisión clave, que profundizaremos con el balance sesgo-varianza en
  la próxima sesión.
- Métricas de regresión: **MAE** (robusto a outliers), **MSE/RMSE**
  (penalizan errores grandes), **$R^2$** (proporción de varianza
  explicada), **MAPE** (error porcentual, falla cerca de cero). Ninguna
  es "la mejor" en absoluto: la elección depende del problema.

---

## 8. El problema de clasificación

En la Sesión 1 trabajamos con **regresión**: la variable respuesta $Y$ era
cuantitativa (un número real, como un precio o un ingreso). En muchos problemas
de negocio, sin embargo, lo que queremos predecir es **cualitativo**: una
**categoría** o **etiqueta de clase**. Por ejemplo:

- ¿Este cliente **pagará** o **no pagará** su crédito? (`good` / `bad`)
- ¿Este correo es **spam** o **no spam**?
- ¿Este tumor es **benigno** o **maligno**?
- ¿Este cliente **se irá** (churn) o **se quedará**?

A este tipo de problema se le llama **problema de clasificación**, y a un modelo
que lo resuelve se le llama **clasificador**. Formalmente, buscamos una función

$$\hat{f}: \mathcal{X} \rightarrow \mathcal{C}$$

que asigna a cada observación $x$ (el vector de variables predictoras) una
etiqueta de clase $c \in \mathcal{C} = \{c_1, c_2, \dots, c_K\}$. Cuando
$K=2$ (por ejemplo `good`/`bad`) hablamos de **clasificación binaria**, que es
el caso que trabajaremos hoy.

### ¿Por qué un clasificador y no un regresor?

Podríamos, en principio, codificar `good = 0` y `bad = 1` y ajustar una
regresión lineal sobre esa variable 0/1. El problema es que:

- La regresión lineal puede predecir valores **fuera del rango [0, 1]**
  (por ejemplo, -0.3 o 1.4), que no tienen interpretación como probabilidad
  ni como clase.
- No captura bien la relación **no lineal** típica entre las variables
  predictoras y la probabilidad de pertenecer a una clase (una relación en
  forma de "S", no de línea recta).
- Las métricas de error de regresión (como el RMSE) no son las que nos
  interesan aquí: nos interesa saber **cuántas veces acertamos la clase**,
  y con qué tipo de errores (falsos positivos vs. falsos negativos), que no
  son intercambiables en la mayoría de aplicaciones de negocio.

Por eso usamos modelos diseñados específicamente para clasificación —hoy
veremos **regresión logística** (que sí modela una probabilidad entre 0 y 1
mediante una función sigmoide) y modelos más flexibles como **Random
Forest**.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import KFold, LeaveOneOut, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Paleta de colores consistente para todos los gráficos de la sesión
COLOR = {
    "blue": "#2a78d6",     # modelo 1 (regresión logística)
    "orange": "#eb6834",   # modelo 2 (random forest)
    "aqua": "#1baf7a",     # modelo 3 (ejercicios)
    "gray": "#898781",     # líneas de referencia / baseline
    "grid": "#e1e0d9",     # grillas
}
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.color"] = COLOR["grid"]
plt.rcParams["axes.edgecolor"] = COLOR["gray"]

## 9. Cargando los datos: German Credit Data

### El problema

Un banco recibe solicitudes de crédito y necesita decidir a quién
aprobarle el préstamo. Quiere un modelo que **prediga el riesgo de que
un solicitante no pague** (mal crédito), para apoyar esa decisión sin
depender solo del criterio manual de un analista.

### El dataset

**German Credit Data**: 1.000 solicitudes de crédito reales, con 20
variables por solicitante (situación de la cuenta corriente, historial
crediticio, propósito del préstamo, monto, antigüedad laboral, edad,
etc.) y una etiqueta `class` (`good`/`bad`) que indica si el crédito
resultó bueno o malo.

Usamos `fetch_openml` de scikit-learn, que descarga (y cachea localmente) el
dataset directamente desde OpenML.

In [ ]:
d = fetch_openml("credit-g", version=1, as_frame=True, parser="auto")
df_credit = d.frame
print("Filas, columnas:", df_credit.shape)
df_credit.head()

In [ ]:
df_credit.info()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
counts = df_credit["class"].value_counts()
ax.bar(counts.index, counts.values, color=[COLOR["blue"], COLOR["orange"]], width=0.5)
for i, v in enumerate(counts.values):
    ax.text(i, v + 8, str(v), ha="center", color="#0b0b0b")
ax.set_title("Distribución de la variable objetivo (class)")
ax.set_ylabel("Número de solicitudes")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

print(counts / counts.sum())

Tenemos **1000 solicitudes**: 700 `good` (70%) y 300 `bad` (30%). Es un
dataset **moderadamente desbalanceado** — algo muy común en riesgo crediticio,
detección de fraude, churn, etc. Este desbalance es justamente la razón por la
que más adelante vamos a preferir la curva **Precision-Recall** sobre la
exactitud simple para evaluar el modelo.

## 10. Preparación de datos: codificación de variables categóricas

La mayoría de los modelos de scikit-learn requieren variables **numéricas**.
El dataset tiene varias columnas categóricas (`checking_status`,
`credit_history`, `purpose`, etc.). Las convertimos a variables *dummy*
(one-hot encoding) con `pd.get_dummies`.

También definimos la variable objetivo: `y_credit = 1` si el crédito es **malo**
(`bad`) y `y_credit = 0` si es **bueno** (`good`). Elegimos `bad` como la "clase
positiva" porque es la clase de interés para el banco (el riesgo que se
quiere detectar) y es, además, la clase minoritaria — el caso típico en
problemas de detección de riesgo.

In [ ]:
y_credit = (df_credit["class"] == "bad").astype(int)
X_credit = pd.get_dummies(df_credit.drop(columns=["class"]), drop_first=True)
print("X:", X_credit.shape, " y positivos (bad):", y_credit.sum(), f"({y_credit.mean():.1%})")

X_train_credit, X_test_credit, y_train_credit, y_test_credit = train_test_split(
    X_credit, y_credit, test_size=0.25, random_state=RANDOM_STATE, stratify=y_credit
)
print("Train:", X_train_credit.shape, " Test:", X_test_credit.shape)

Nota: usamos `stratify=y_credit` para que la proporción de `bad`/`good` se
mantenga aproximadamente igual en train y test — algo especialmente
importante cuando las clases están desbalanceadas.

## 11. Entrenando dos modelos: uno simple, uno más flexible

Vamos a entrenar dos clasificadores con la misma información y comparar su
comportamiento durante toda la sesión:

- **Regresión logística** (dentro de un `Pipeline` con estandarización de
  variables): un modelo lineal, simple e interpretable. Alto sesgo, baja
  varianza.
- **Random Forest**: un ensamble de árboles de decisión, mucho más flexible.
  Menor sesgo, pero mayor varianza (más propenso a sobreajustar si no se
  controla su complejidad).

In [ ]:
logreg = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])
logreg.fit(X_train_credit, y_train_credit)

rf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE)
rf.fit(X_train_credit, y_train_credit)

print("Modelos entrenados:", logreg.named_steps["clf"].__class__.__name__, "y", rf.__class__.__name__)

## 12. Métricas de desempeño para clasificación

### Matriz de confusión

La **matriz de confusión** cruza la clase real con la clase predicha. Para
clasificación binaria, con la clase positiva = `bad` (1):

|                     | Predicho: bueno (0) | Predicho: malo (1) |
|---------------------|:---:|:---:|
| **Real: bueno (0)** | TN (Verdadero Negativo) | FP (Falso Positivo) |
| **Real: malo (1)**  | FN (Falso Negativo) | TP (Verdadero Positivo) |

- **TP** (True Positive): predijimos `bad` y sí era `bad`.
- **TN** (True Negative): predijimos `good` y sí era `good`.
- **FP** (False Positive): predijimos `bad` pero en realidad era `good` — un
  "error tipo I" (rechazamos un buen cliente).
- **FN** (False Negative): predijimos `good` pero en realidad era `bad` — un
  "error tipo II" (aprobamos un mal cliente). En riesgo de crédito, este
  suele ser el error más costoso.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for ax, model, name in zip(axes, [logreg, rf], ["Regresión logística", "Random Forest"]):
    ConfusionMatrixDisplay.from_estimator(
        model, X_test_credit, y_test_credit, display_labels=["good (0)", "bad (1)"],
        cmap="Blues", colorbar=False, ax=ax,
    )
    ax.set_title(name)
plt.tight_layout()
plt.show()

### Exactitud, precisión, recall y F-score

A partir de TP, FP, FN, TN se derivan las métricas más usadas:

$$\text{Exactitud (Accuracy)} = \frac{TP + TN}{TP + TN + FP + FN}$$

$$\text{Precisión} = \frac{TP}{TP + FP} \qquad \text{(de lo que predije como positivo, ¿cuánto acerté?)}$$

$$\text{Recall (Sensibilidad)} = \frac{TP}{TP + FN} \qquad \text{(de todos los positivos reales, ¿cuántos detecté?)}$$

$$F_1 = 2 \cdot \frac{\text{Precisión} \cdot \text{Recall}}{\text{Precisión} + \text{Recall}} \qquad \text{(media armónica de precisión y recall)}$$

Con clases desbalanceadas, la **exactitud puede ser engañosa**: un modelo que
siempre predice `good` acertaría el 70% de las veces sin haber aprendido
nada útil sobre el riesgo. Por eso miramos precisión, recall y F1 en
conjunto — normalmente enfocados en la clase positiva (`bad`).

In [ ]:
def resumen_metricas(model, X_test_credit, y_test_credit, nombre):
    y_pred = model.predict(X_test_credit)
    return {
        "modelo": nombre,
        "accuracy": accuracy_score(y_test_credit, y_pred),
        "precision (bad)": precision_score(y_test_credit, y_pred),
        "recall (bad)": recall_score(y_test_credit, y_pred),
        "f1 (bad)": f1_score(y_test_credit, y_pred),
    }

tabla = pd.DataFrame([
    resumen_metricas(logreg, X_test_credit, y_test_credit, "Regresión logística"),
    resumen_metricas(rf, X_test_credit, y_test_credit, "Random Forest"),
]).set_index("modelo").round(3)
tabla

Observa que ambos modelos tienen accuracy relativamente alta (cercana al
70-75%, que ya era la proporción de `good`), pero el **recall de la clase
`bad`** — la que de verdad nos importa detectar — es mucho más bajo. Este es
exactamente el efecto del desbalance de clases sobre la accuracy.

### Curva ROC y AUC

Hasta ahora evaluamos el modelo con un umbral fijo de 0.5 sobre la
probabilidad predicha. La curva **ROC** (Receiver Operating Characteristic)
muestra el desempeño del modelo **para todos los posibles umbrales** a la vez,
graficando:

- Eje X: **FPR** (False Positive Rate) $= \frac{FP}{FP+TN}$ — tasa de falsos positivos.
- Eje Y: **TPR** (True Positive Rate) $=$ Recall $= \frac{TP}{TP+FN}$.

Un clasificador aleatorio traza la diagonal (AUC = 0.5). Mientras más se
acerque la curva a la esquina superior izquierda, mejor. El **AUC** (área
bajo la curva) resume esto en un solo número: la probabilidad de que el
modelo asigne una probabilidad más alta a un caso positivo elegido al azar
que a uno negativo elegido al azar.

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
RocCurveDisplay.from_estimator(logreg, X_test_credit, y_test_credit, name="Regresión logística", curve_kwargs={"color": COLOR["blue"]}, ax=ax)
RocCurveDisplay.from_estimator(rf, X_test_credit, y_test_credit, name="Random Forest", curve_kwargs={"color": COLOR["orange"]}, ax=ax)
ax.plot([0, 1], [0, 1], linestyle="--", color=COLOR["gray"], label="Aleatorio (AUC=0.5)")
ax.set_title("Curva ROC — comparación de modelos")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

### Curva Precision-Recall

Cuando las clases están **desbalanceadas** (como aquí: 70/30), la curva ROC
puede ser demasiado optimista, porque el FPR se calcula sobre la clase
negativa (mayoritaria), que es "fácil" de mantener baja. La curva
**Precision-Recall** (Precisión vs. Recall, ambas calculadas sobre la clase
positiva) es más informativa en estos casos, porque no involucra los
verdaderos negativos en absoluto.

La línea base de un clasificador aleatorio en esta curva no es la diagonal,
sino una línea horizontal en `precisión = proporción de positivos` (aquí,
0.30).

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
PrecisionRecallDisplay.from_estimator(logreg, X_test_credit, y_test_credit, name="Regresión logística", curve_kwargs={"color": COLOR["blue"]}, ax=ax)
PrecisionRecallDisplay.from_estimator(rf, X_test_credit, y_test_credit, name="Random Forest", curve_kwargs={"color": COLOR["orange"]}, ax=ax)
baseline = y_test_credit.mean()
ax.axhline(baseline, linestyle="--", color=COLOR["gray"], label=f"Aleatorio (precisión={baseline:.2f})")
ax.set_title("Curva Precision-Recall — comparación de modelos")
ax.legend(loc="upper right")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

### Curvas de Lift y Ganancia Acumulada (Cumulative Gain)

Estas curvas responden una pregunta muy práctica para negocio: *si ordeno a
todos los clientes de mayor a menor probabilidad predicha, y solo puedo
"tratar" (revisar manualmente, contactar, etc.) al x% con mayor probabilidad,
¿cuántos de los casos positivos reales estoy capturando, y qué tan mejor es
eso comparado con elegir al azar?*

- **Ganancia Acumulada (Cumulative Gain)**: eje X = % de la población
  contactada (ordenada por probabilidad predicha, de mayor a menor), eje Y =
  % acumulado de positivos reales capturados. La línea base aleatoria es la
  diagonal.
- **Lift**: es la ganancia acumulada dividida por la proporción de datos
  usada, es decir, cuántas veces mejor que el azar es el modelo en ese punto
  de corte. La línea base aleatoria es una horizontal en 1.

No vienen implementadas directamente en scikit-learn, pero son sencillas de
calcular manualmente:

In [ ]:
def curva_ganancia_lift(y_true, y_score):
    orden = np.argsort(-y_score)
    y_ordenado = np.asarray(y_true)[orden]
    n = len(y_ordenado)
    positivos_totales = y_ordenado.sum()

    pct_poblacion = np.arange(1, n + 1) / n
    ganancia_acumulada = np.cumsum(y_ordenado) / positivos_totales
    lift = ganancia_acumulada / pct_poblacion
    return pct_poblacion, ganancia_acumulada, lift


fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

for model, name, color in [(logreg, "Regresión logística", COLOR["blue"]), (rf, "Random Forest", COLOR["orange"])]:
    proba = model.predict_proba(X_test_credit)[:, 1]
    pct, ganancia, lift = curva_ganancia_lift(y_test_credit, proba)
    axes[0].plot(pct, ganancia, label=name, color=color, linewidth=2)
    axes[1].plot(pct, lift, label=name, color=color, linewidth=2)

axes[0].plot([0, 1], [0, 1], linestyle="--", color=COLOR["gray"], label="Aleatorio")
axes[0].set_xlabel("% de la población contactada (ordenada por score)")
axes[0].set_ylabel("% acumulado de casos 'bad' capturados")
axes[0].set_title("Curva de Ganancia Acumulada")
axes[0].legend()
axes[0].spines[["top", "right"]].set_visible(False)

axes[1].axhline(1.0, linestyle="--", color=COLOR["gray"], label="Aleatorio (lift=1)")
axes[1].set_xlabel("% de la población contactada (ordenada por score)")
axes[1].set_ylabel("Lift")
axes[1].set_title("Curva de Lift")
axes[1].legend()
axes[1].spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

**Lectura práctica**: si el banco solo tiene capacidad para revisar
manualmente al 20% de las solicitudes con mayor score de riesgo, la curva de
ganancia acumulada nos dice directamente qué porcentaje de los créditos
realmente malos estaría capturando en ese 20% — y la curva de lift nos dice
cuántas veces mejor es eso que revisar un 20% al azar.

## Resumen de conceptos clave — Clasificación

- Un problema de **clasificación** busca asignar una etiqueta de clase (no un
  número continuo); usamos modelos como regresión logística o Random Forest,
  no regresión lineal.
- La **matriz de confusión** (TP, FP, FN, TN) es la base de todas las
  métricas de clasificación: exactitud, precisión, recall y F1.
- La **exactitud puede engañar** en datasets desbalanceados — por eso
  miramos precisión/recall/F1 de la clase de interés, la curva **ROC**
  (y su AUC), y sobre todo la curva **Precision-Recall** cuando hay
  desbalance.
- Las curvas de **Lift** y **Ganancia Acumulada** traducen el desempeño del
  modelo a una pregunta de negocio directa: *"¿cuánto mejor que el azar soy
  si solo puedo actuar sobre el x% con mayor score?"*

### Próxima sesión

En la Sesión 2 veremos el balance **sesgo-varianza**, sobreajuste/subajuste,
**validación cruzada**, **árboles de decisión** y **métodos de ensamble**
(bagging, boosting) — retomando los modelos y datos de crédito de hoy antes
de pasar a un dataset nuevo.

---

# Ejercicio

Las partes 1-2 se resuelven sobre el modelo y los datos de regresión ya
cargados arriba (Ames Housing). La parte 3 se resuelve sobre los modelos y
datos de clasificación (German Credit, `logreg`/`rf`,
`X_train_credit`/`y_train_credit`).

| # | Parte | Tiempo sugerido |
|---|---|---|
| 1 | Efecto de los outliers en las métricas (regresión) | 10 min |
| 2 | Gráfico de residuales (regresión) | 10 min |
| 3 | Comparar un tercer modelo con curvas ROC (clasificación) | 10 min |

Si una parte se atasca, pasen a la siguiente: valen más las tres intentadas que una perfecta.

### Ejercicio 1 — Efecto de los outliers en las métricas

El dataset tiene casas muy costosas (outliers en `price`) y casas con
áreas atípicamente grandes. Filtra el conjunto de datos eliminando el
1% de observaciones con `price` más alto, vuelve a partir train/val/test
con la misma semilla, reentrena la Regresión Lineal, y compara las
métricas MAE, RMSE y $R^2$ antes y después de filtrar. ¿Qué métrica
cambia más? ¿Por qué?

In [ ]:
# TODO: filtra el 1% de precios más altos, reentrena la regresión lineal
# y compara MAE/RMSE/R2 contra el modelo original (sin filtrar).

### Ejercicio 2 — Gráfico de residuales

Los **residuales** son la diferencia entre el valor real y el
predicho: $e_i = y_i - \hat{y}_i$. Grafica los residuales del modelo de
Regresión Lineal (eje X: valor predicho $\hat{y}_i$, eje Y: residual
$e_i$) sobre el conjunto de validation, y traza una línea horizontal en
cero. ¿Qué patrón observas? ¿Los residuales se ven distribuidos de
forma aleatoria alrededor de cero, o hay una tendencia?

In [ ]:
# TODO: grafica los residuales (y_val - pred_lineal_val) contra
# pred_lineal_val, con una línea horizontal en 0. Interpreta el patrón.

### Ejercicio 3 — Comparar un tercer modelo con curvas ROC

Entrena un tercer modelo (por ejemplo, un `DecisionTreeClassifier` con
`max_depth=5` o un `SVC(probability=True)`) sobre `X_train_credit`/`y_train_credit`.
Grafica su curva ROC junto a las de `logreg` y `rf` (puedes reutilizar
`RocCurveDisplay.from_estimator` pasando el mismo `ax`). Compara los AUC de
los tres modelos: ¿cuál es mayor? ¿Coincide con el modelo que consideras
"mejor" en la práctica, considerando también la interpretabilidad?

In [ ]:
# TODO: 1) entrena un tercer modelo (ej. DecisionTreeClassifier(max_depth=5) o SVC(probability=True))
# TODO: 2) usa RocCurveDisplay.from_estimator para graficar su curva ROC junto a las de logreg y rf
# TODO: 3) compara los AUC de los tres modelos y concluye cuál preferirías y por qué

---

### Ejercicio extra (opcional, sin cronometrar) — Comparar un subconjunto distinto de variables

Entrena una nueva Regresión Lineal usando **solo** estas variables:
`area`, `Year.Built` y `Lot.Area` (sin `Overall.Qual`, `Total.Bsmt.SF`,
etc.). Compara el $R^2$ en validation contra el modelo original (con
las 9 variables). ¿Qué te dice esta comparación sobre la importancia de
`Overall.Qual` para explicar el precio?

In [ ]:
# TODO: entrena una LinearRegression con el subconjunto de variables
# ['area', 'Year.Built', 'Lot.Area'] y compara su R2 en validation
# contra el modelo con las 9 variables originales.